In [1]:
import numpy as np

from line_solver import *

In [2]:
model = Network('model')

node = np.empty(4, dtype=object)
node[0] = Delay(model, 'Delay')
node[1] = Queue(model, 'Queue1', SchedStrategy.PS)
node[2] = Source(model,'Source')
node[3] = Sink(model,'Sink')

jobclass = np.empty(2, dtype=object)
jobclass[0] = ClosedClass(model, 'ClosedClass', 2, node[0], 0)
jobclass[1] = OpenClass(model, 'OpenClass', 0)

node[0].setService(jobclass[0], Erlang(3,2))
node[0].setService(jobclass[1], HyperExp(0.5,3.0,10.0))

node[1].setService(jobclass[0], HyperExp(0.1,1.0,10.0))
node[1].setService(jobclass[1], Exp(1))

node[2].setArrival(jobclass[1], Exp(0.1))

K = model.getNumberOfClasses()

P = model.initRoutingMatrix()
pmatrix = np.empty(K, dtype=object)
pmatrix[0] = [[0.0,1.0,0.0,0.0],[1.0,0.0,0.0,0.0], [0.0,0.0,0.0,0.0], [0.0,0.0,0.0,0.0]]
pmatrix[1] = [[0.0,1.0,0.0,0.0], [0.0,0.0,0.0,1.0], [1.0,0.0,0.0,0.0], [0.0,0.0,0.0,0.0]]
P.setRoutingMatrix(jobclass, node, pmatrix)
            
model.link(P);


In [3]:
solver = np.array([], dtype=object)
#solver = np.append(solver, SolverCTMC(model,'keep',True))
solver = np.append(solver, SolverJMT(model,'seed',23000,'verbose',True,'keep',True))
#solver = np.append(solver, SolverSSA(model,'seed',23000,'verbose',True,'samples',50000))
solver = np.append(solver, SolverFluid(model))
solver = np.append(solver, SolverMVA(model))
#solver = np.append(solver, SolverNC(model,'exact'))
#solver = np.append(solver, SolverMAM(model))
#np.append(solver, LINE(model))

AvgTable = np.empty(len(solver), dtype=object)
for s in range(len(solver)):
    print(f'\nSOLVER: {solver[s].getName()}')
    AvgTable[s] = solver[s].getAvgTable()
    print(AvgTable[s])



SOLVER: SolverJMT
JMT Model: /tmp/workspace/jsim/2475437434169090594/model.jsim
JMT Command: java -cp /home/gcasale/Dropbox/code/line-solver.git/python/line_solver/JMT.jar jmt.commandline.Jmt sim /tmp/workspace/jsim/2475437434169090594/model.jsim -seed 23000 --illegal-access=permit
JMT Analysis completed. Runtime: 1.899 seconds
  Station     JobClass      QLen      Util     RespT    ResidT      Tput
0   Delay  ClosedClass  1.452831  1.452831  0.666437  0.666437  2.153604
1   Delay    OpenClass  0.021961  0.021961  0.215814  0.215814  0.100006
2  Queue1  ClosedClass  0.547169  0.402157  0.259830  0.259830  2.160204
3  Queue1    OpenClass  0.163907  0.095781  1.711018  1.711018  0.100006
5  Source    OpenClass  0.000000  0.000000  0.000000  0.000000  0.100006

SOLVER: SolverFluid
  Station     JobClass      QLen      Util     RespT    ResidT      Tput
0   Delay  ClosedClass  1.556420  1.556420  0.666667  0.666667  2.334630
1   Delay    OpenClass  0.021621  0.021621  0.217205  0.217205  